# 🎙️ AI Friend: Voice Cloning Training (GPT-SoVITS)

<a href="https://colab.research.google.com/github/PALabs-v1/AI_friend/blob/main/notebooks/ai_friend_voice_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Trains a custom **GPT-SoVITS** voice-clone model for the `voice_agent` /
`local_voice` runtime described in `docker-compose.infra.yml`. This is the
heaviest GPU job in the project — GPT-SoVITS's Colab image is CUDA-oriented
and the fine-tuning step alone needs a real GPU, which is why this lives here
rather than as a local script (see `docker-compose.heavy.yml`'s comment on
why GPT-SoVITS is excluded from the CPU-safe compose layering).

**What this produces:** two weight files (`ai_friend_voice.ckpt`,
`ai_friend_voice.pth`) plus a vocoder checkpoint that replace the bundled
default voice (`backend/assets/voice/default_voice.wav`, Phase 1.1) with a
voice cloned from *your* training data. This is a different, heavier step
than `backend/scripts/audio/record_voice.py` (Phase 2.3) — that script
records an 8-second **reference clip** GPT-SoVITS clones zero-shot from at
inference time; this notebook actually **fine-tunes the model's weights** on
10+ minutes of your audio for a stronger, more stable clone. Both feed the
same runtime; neither requires the other.

### Quick instructions
1. Run **Cell 1** (GPU check) — refuse to continue if it reports no GPU.
   `Runtime → Change runtime type → T4 GPU` (or better) if it does.
2. (Recommended) Run **Cell 2** to mount Google Drive, so your dataset and
   trained weights survive a Colab disconnect — training can take well over
   an hour and free-tier Colab sessions are not guaranteed to stay alive.
3. Run **Cell 3** (the Launcher). If a popup says "Restart Session", click it
   and run the cell again — this is normal after the CUDA/torch reinstall.
4. Wait for the **Gradio public link** to appear and open it. Follow
   `../docs/COLAB_PATHS_CHEATSHEET.md` for the exact tab-by-tab WebUI
   workflow (slicing → ASR → formatting → training → inference) — that
   document is the detailed manual; this notebook is the environment it
   assumes.
5. Once you've picked a checkpoint pair you're happy with in the WebUI's
   inference tab, come back here and run the **export cell** at the bottom
   to package everything into one zip named and pathed exactly the way the
   local runtime expects.

See `notebooks/README.md` in this folder for the full walkthrough,
including what "good" training data looks like and how to install the
result locally.

In [ ]:
# Cell 1 -- GPU check. GPT-SoVITS's training step is impractical on CPU;
# do not proceed past this cell without a GPU runtime attached.
import subprocess

try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(out.stdout)
    print("GPU detected -- safe to continue.")
except (subprocess.CalledProcessError, FileNotFoundError):
    raise RuntimeError(
        "No GPU visible. Go to Runtime -> Change runtime type -> select a "
        "GPU (T4 is enough), then re-run this cell before continuing."
    )

### Cell 2 -- Mount Google Drive (recommended, optional)

Training data and output checkpoints land under `/content/drive/MyDrive/`
instead of the ephemeral Colab VM disk, so a disconnect mid-training doesn't
cost you the run. Skip this only for a quick smoke test you don't mind
redoing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

WORKSPACE = "/content/drive/MyDrive/ai_friend_voice_training"
os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(os.path.join(WORKSPACE, "training_data"), exist_ok=True)
print(f"Workspace: {WORKSPACE}")
print(
    "Upload your raw recordings into training_data/ via the Colab file "
    "sidebar before Phase 1 of docs/COLAB_PATHS_CHEATSHEET.md, or run "
    "without Drive and upload directly to /content/training_data instead."
)

In [ ]:
# Cell 3 -- Launcher: Setup & Start WebUI
import os

# 1. Colab Navigation & Workspace Prep
%cd /content
if not os.path.exists("GPT-SoVITS"):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS

# 2. Install Linux system tools & Python libraries
!apt-get install -y ffmpeg cmake libsox-dev
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2.0" numba librosa==0.10.2 gradio==4.44.1 ffmpeg-python onnxruntime-gpu

# The opencc pin in GPT-SoVITS's own requirements.txt is frequently broken on
# a fresh Colab image; force the reimplementation instead of the upstream one.
!pip uninstall -y opencc
!pip install --no-binary=opencc opencc-python-reimplemented
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 3. Skip the broken opencc pin from upstream requirements.txt
!grep -v "opencc" requirements.txt > req_colab.txt
!pip install -r req_colab.txt

# 4. Download the V4 pretrained base weights (skipped if already present --
#    re-running this cell after a "Restart Session" popup won't re-download).
!mkdir -p GPT_SoVITS/pretrained_models/gsv-v4-pretrained
!mkdir -p GPT_SoVITS/pretrained_models/chinese-hubert-base
!mkdir -p GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large

if not os.path.exists("GPT_SoVITS/pretrained_models/s1v3.ckpt"):
    !wget "https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s1v3.ckpt" -O GPT_SoVITS/pretrained_models/s1v3.ckpt
if not os.path.exists("GPT_SoVITS/pretrained_models/gsv-v4-pretrained/s2Gv4.pth"):
    !wget "https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v4-pretrained/s2Gv4.pth" -O GPT_SoVITS/pretrained_models/gsv-v4-pretrained/s2Gv4.pth
if not os.path.exists("GPT_SoVITS/pretrained_models/gsv-v4-pretrained/vocoder.pth"):
    !wget "https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v4-pretrained/vocoder.pth" -O GPT_SoVITS/pretrained_models/gsv-v4-pretrained/vocoder.pth

if not os.path.exists(
    "GPT_SoVITS/pretrained_models/chinese-hubert-base/pytorch_model.bin"
):
    !wget https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/chinese-hubert-base/config.json -O GPT_SoVITS/pretrained_models/chinese-hubert-base/config.json
    !wget https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/chinese-hubert-base/pytorch_model.bin -O GPT_SoVITS/pretrained_models/chinese-hubert-base/pytorch_model.bin

if not os.path.exists(
    "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/pytorch_model.bin"
):
    !wget https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/chinese-roberta-wwm-ext-large/config.json -O GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/config.json
    !wget https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/chinese-roberta-wwm-ext-large/tokenizer.json -O GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/tokenizer.json
    !wget https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/chinese-roberta-wwm-ext-large/pytorch_model.bin -O GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/pytorch_model.bin

# 5. If Drive is mounted (Cell 2), symlink its training_data/ in so the WebUI
#    sees it at the path docs/COLAB_PATHS_CHEATSHEET.md's Phase 1 expects.
_drive_data = "/content/drive/MyDrive/ai_friend_voice_training/training_data"
if os.path.exists(_drive_data) and not os.path.exists("/content/training_data"):
    os.symlink(_drive_data, "/content/training_data")
    print(f"Linked {_drive_data} -> /content/training_data")
elif not os.path.exists("/content/training_data"):
    os.makedirs("/content/training_data", exist_ok=True)
    print(
        "No Drive data found -- upload recordings into /content/training_data "
        "via the file sidebar now."
    )

# 6. Launch (Gradio public link)
os.environ["is_share"] = "True"
!python webui.py

---
## Export for the local runtime

Run this **after** you've trained and picked a checkpoint pair in the WebUI's
inference tab above (Phase 5 of `docs/COLAB_PATHS_CHEATSHEET.md`). It renames
your chosen `.ckpt`/`.pth` to the names `docker-compose.infra.yml`'s
`gpt-sovits` service and `.env.example`'s `CUSTOM_GPT_PATH`/
`CUSTOM_SOVITS_PATH` expect by default, and zips them with the vocoder for a
single download.

In [ ]:
# Export cell -- run after training. Edit the two filenames below to match
# whichever checkpoints you selected in the WebUI's 1C-Inference sub-tab.
import glob
import os
import shutil
import zipfile

GPT_CKPT_NAME = (
    None  # e.g. "ai_friend_voice-e15.ckpt" -- leave None to auto-pick the newest
)
SOVITS_CKPT_NAME = None  # e.g. "ai_friend_voice_e8.pth"

export_dir = "/content/voice_export"
os.makedirs(export_dir, exist_ok=True)


def _newest(pattern):
    matches = sorted(glob.glob(pattern), key=os.path.getmtime)
    if not matches:
        raise FileNotFoundError(f"No files matched {pattern} -- did training finish?")
    return matches[-1]


gpt_src = (
    os.path.join("/content/GPT-SoVITS/GPT_weights", GPT_CKPT_NAME)
    if GPT_CKPT_NAME
    else _newest("/content/GPT-SoVITS/GPT_weights/*.ckpt")
)
sovits_src = (
    os.path.join("/content/GPT-SoVITS/SoVITS_weights", SOVITS_CKPT_NAME)
    if SOVITS_CKPT_NAME
    else _newest("/content/GPT-SoVITS/SoVITS_weights/*.pth")
)
vocoder_src = (
    "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/gsv-v4-pretrained/vocoder.pth"
)

# Names the local .env.example / docker-compose.infra.yml default paths expect:
#   CUSTOM_GPT_PATH=GPT_weights/ai_friend_voice.ckpt
#   CUSTOM_SOVITS_PATH=SoVITS_weights/ai_friend_voice.pth
shutil.copy(gpt_src, os.path.join(export_dir, "ai_friend_voice.ckpt"))
shutil.copy(sovits_src, os.path.join(export_dir, "ai_friend_voice.pth"))
shutil.copy(vocoder_src, os.path.join(export_dir, "vocoder.pth"))

zip_path = "/content/ai_friend_voice_export.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ("ai_friend_voice.ckpt", "ai_friend_voice.pth", "vocoder.pth"):
        zf.write(os.path.join(export_dir, fname), fname)

print(
    f"Wrote {zip_path} from:\n  GPT:    {gpt_src}\n  SoVITS: {sovits_src}\n  Vocoder: {vocoder_src}"
)

try:
    from google.colab import files

    files.download(zip_path)
except ImportError:
    print("Not running in Colab -- find the zip at", zip_path)

### Installing the export locally

Unzip `ai_friend_voice_export.zip` at the repo root into:

| File | Local path |
| :--- | :--- |
| `ai_friend_voice.ckpt` | `models/GPT_weights/ai_friend_voice.ckpt` |
| `ai_friend_voice.pth` | `models/SoVITS_weights/ai_friend_voice.pth` |
| `vocoder.pth` | `models/SoVITS_weights/vocoder.pth` |

These are the exact paths `docker-compose.infra.yml`'s `gpt-sovits` service
volume-mounts (`./models/GPT_weights` / `./models/SoVITS_weights`) and the
defaults `CUSTOM_GPT_PATH`/`CUSTOM_SOVITS_PATH` in `.env.example` point at --
no `.env` edit needed unless you renamed the files. Restart the `gpt-sovits`
container (`docker compose -f docker-compose.infra.yml up -d --force-recreate gpt-sovits`)
to pick them up; `sovits_healthcheck.sh` will report unhealthy until it can
actually load them, so check `docker compose logs local_voice` if it doesn't
come up within a few minutes -- first load of a real checkpoint is
memory-heavy and can take minutes on CPU fallback.

Before promoting over your current voice, run the A/B gate in
`docs/COLAB_PATHS_CHEATSHEET.md`'s Phase 7 (Validation and Safe Promotion) --
keep the previous `.ckpt`/`.pth` pair around to roll back to.